In [1]:
!pip install -U langchain-text-splitters
!pip install -U langchain-community langchain-text-splitters langchain
!pip install pypdf
!pip install -U langchain-huggingface
!pip install -qU langchain-chroma
!pip install -qU langchain
!pip install -qU langchain-groq
!pip install sentence-transformers
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━

In [2]:
from langchain_groq import ChatGroq
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
import shutil
import gradio as gr
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import re
import gc
import shutil
from google.colab import drive
from google.colab import userdata

In [14]:
# --- CHEMINS ---

drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Smart-HR-Assistant"
NEW_DATA_PATH = os.path.join(PROJECT_PATH, "nouveau_data")
ACTUEL_DATA_PATH = os.path.join(PROJECT_PATH, "actuel_data")
CHROMA_PATH = os.path.join(PROJECT_PATH, "chroma_db")


# Création des dossiers si nécessaire
for path in [NEW_DATA_PATH, ACTUEL_DATA_PATH, CHROMA_PATH]:
    os.makedirs(path, exist_ok=True)

# --- CONFIGURATION API ---
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm_chat = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

llm_multi_query = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Le rédacteur
instruction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Tu es un expert en analyse de documents RH. Ton rôle est de fournir des réponses "
        "précises et structurées à partir du contexte fourni ci-dessous.\n\n"
        "CONTEXTE :\n{context}\n\n"
        "DIRECTIVES DE PRÉCISION :\n"
        "1. SYNTHÈSE : Synthétise les points clés (noms, dates, montants).\n"
        "2. SOURCES : Pour chaque information importante, tu DOIS citer le document source. "
        "Exemple : 'La prime est de 300€ [Source: politique_primes.pdf]'.\n"
        "3. INTÉGRITÉ : Si l'info n'est pas dans le contexte, dis 'Je ne trouve pas cette info dans les documents fournis'.\n"
        "4. ANTI-MÉLANGE : Si deux documents se contredisent, précise-le (ex: 'Le doc A dit X, mais le doc B dit Y').\n"
        "5. FORMAT : Utilise des listes à puces et du gras pour les chiffres."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

In [7]:
def get_or_update_db(update=True):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # 1. Charger la base existante (ou en créer une vide)
    vector_db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

    # 2. Vérifier s'il y a du nouveau
    if update:
      new_files = [f for f in os.listdir(NEW_DATA_PATH) if f.endswith('.pdf')]

      if new_files:
          loader = PyPDFDirectoryLoader(NEW_DATA_PATH)
          new_docs = loader.load()

          text_splitter = RecursiveCharacterTextSplitter(
              chunk_size=800,
              chunk_overlap=200,
              length_function=len,
              add_start_index=True
          )

          new_chunks = text_splitter.split_documents(new_docs)

          # Ajout à Chroma
          vector_db.add_documents(new_chunks)

          # Déplacement vers l'archive
          for file_name in new_files:
              shutil.move(os.path.join(NEW_DATA_PATH, file_name),
                          os.path.join(ACTUEL_DATA_PATH, file_name))
              print(f"Archivé : {file_name}")
      else:
          print("Aucun nouveau fichier. Utilisation de la base existante.")

    return vector_db

In [41]:
def vider_db():
    global vector_db
    # Vider la collection
    try:
        vector_db.delete_collection()
        print("Collection vidée.")

    except Exception as e:
        print(f"Erreur lors du vidage : {e}")

    # Supprimer la collection
    for nom in os.listdir(CHROMA_PATH):
        item_path = os.path.join(path, nom)
        if os.path.isdir(item_path):
            try:
                shutil.rmtree(item_path)
                print(f"Dossier de collection supprimé : {nom}")
            except Exception as e:
                print(f"Impossible de supprimer {nom} : {e}")

    # Supprimer les fichiers PDF dans actuel_data
    for nom in os.listdir(ACTUEL_DATA_PATH):
        chemin_complet = os.path.join(ACTUEL_DATA_PATH, nom)
        try:
            if os.path.isfile(chemin_complet):
                os.unlink(chemin_complet)
            elif os.path.isdir(chemin_complet):
                shutil.rmtree(chemin_complet)
            print("Actuel data vidé")
        except Exception as e:
            print(f"Impossible de supprimer {nom} : {e}")
    return "✅ La base de données a été vidée avec succès."

In [40]:
def respond(message, chat_history):
    global llm_chat, llm_multi_query, vector_db

    # --- TRANSFORME L'HISTORIQUE POUR LE LLM ---
    formatted_history = []
    for user_msg, ai_msg in chat_history:
        formatted_history.append(HumanMessage(content=user_msg))
        formatted_history.append(AIMessage(content=ai_msg))

    # --- ÉTAPE 2 : RECHERCHE AVANCÉE (Multi-Query) ---
    advanced_retriever = MultiQueryRetriever.from_llm(
        retriever=vector_db.as_retriever(search_kwargs={"k": 5}),
        llm=llm_multi_query
    )

    # Le Multi-Query va générer des variantes pour mieux fouiller la DB
    docs = advanced_retriever.invoke(input=message)
    context = "\n\n".join([d.page_content for d in docs])

    # --- ÉTAPE 3 : RÉPONSE FINALE (Le Rédacteur) ---
    chain = instruction_prompt | llm_chat
    result = chain.invoke({
        "context": context,
        "question": message,
        "chat_history": formatted_history
    })

    # 4. Ajouter l'échange à l'historique
    chat_history.append((message, result.content))
    return "", chat_history

In [39]:
def upload_files(files):
    global vector_db
    # 1. Sécurité anti-vide
    if not files:
        gr.Warning("⚠️ Veuillez sélectionner au moins un fichier.")
        return "ERREUR : Aucun fichier sélectionné."

    # récuperer les noms de fichiers déjà archivés
    actuel_files = set(os.listdir(ACTUEL_DATA_PATH))

    files_to_process = []
    doublons_count = 0

    for file in files:
        # extraire le nom propre du fichier (ex: "charte.pdf")
        nom_fichier = os.path.basename(file.name)

        # COMPARAISON : vérifier si le NOM est déjà dans le SET
        if nom_fichier not in actuel_files:
            destination = os.path.join(NEW_DATA_PATH, nom_fichier)
            shutil.copy(file.name, destination)
            files_to_process.append(nom_fichier)
        else:
            doublons_count += 1

    # 2. ne lancer que s'il y a du nouveau fichier
    if files_to_process:
        get_or_update_db(update=True)
        msg = f"✅ SUCCÈS : {len(files_to_process)} fichiers ajoutés."
        if doublons_count > 0:
            msg += f" ({doublons_count} doublons ignorés)"
        return msg
    else:
        return "ℹ️ Tous ces fichiers sont déjà dans la base. Aucun nouveau fichier à synchroniser."

In [15]:
import base64
PATH_TO_ICON = os.path.join(PROJECT_PATH, "icon/chatbot.png")

def get_base64_image(image_path):
    if os.path.exists(image_path):
        with open(image_path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    else:
        print(f"⚠️ Erreur : Fichier non trouvé à l'adresse {image_path}")
        return ""


In [16]:
img_data = get_base64_image(PATH_TO_ICON)

In [13]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.HTML(f"""
        <div style="display: flex; align-items: center; gap: 15px; padding: 10px;">
            <img src="data:image/png;base64,{img_data}" width="50px">
            <h1 style="margin: 0; font-family: sans-serif;">Assistant RH Intelligent</h1>
        </div>
    """)

    gr.Markdown("L'IA qui connaît vos documents internes sur le bout des doigts.")

    with gr.Tabs():
        # --- ONGLET 1 : CHAT ---
        with gr.Tab("💬 Chat avec l'Expert"):
            chatbot = gr.Chatbot(label="Discussion", height=500)
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Posez votre question (ex: Quelles sont les règles de mutuelle ?)",
                    show_label=False,
                    scale=9
                )
                submit_btn = gr.Button("Envoyer", variant="primary", scale=1)

            clear_chat = gr.ClearButton([msg, chatbot], value="Effacer la conversation")

        # --- ONGLET 2 : ADMINISTRATION ---
        with gr.Tab("⚙️ Administration"):
            gr.Markdown("### 📂 Gestion des documents")
            gr.Markdown("Ajoutez de nouveaux PDF pour mettre à jour la base de connaissances.")

            with gr.Row():
                upload_btn = gr.File(
                    label="Déposez vos PDF ici",
                    file_count="multiple",
                    file_types=[".pdf"]
                )

            # Bouton de validation principal
            process_btn = gr.Button("📥 Ajouter à la base de données", variant="primary")

            # Afficheur de statut
            status_label = gr.Textbox(
                              label="Statut du traitement",
                              placeholder="Les étapes de l'indexation s'afficheront ici...",
                              lines=10,
                              max_lines=15,
                              interactive=False,
                              show_copy_button=True
                          )
            gr.HTML("<br><br><br>") # Un peu d'espace avant la zone de danger
            gr.Markdown("---")

            # Zone de danger en bas à gauche
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("🔒 **Maintenance**")
                    clear_db_btn = gr.Button("🗑️ Vider la DB", variant="stop", size="sm")
                with gr.Column(scale=4):
                    # Cette colonne vide sert à pousser le bouton vers la gauche
                    pass

    # --- LOGIQUE DES BOUTONS ---

    # Envoi du message
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])

    # Indexation des fichiers
    process_btn.click(
        fn=upload_files,
        inputs=upload_btn,
        outputs=status_label,
        show_progress="full"
    )

    # Vidage de la base
    clear_db_btn.click(
        fn=vider_db,
        outputs=status_label,
        show_progress="full"
    )

# Lancement de l'application
if __name__ == "__main__":
    demo.launch(debug=True)

/tmp/ipykernel_4300/128250576.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_4300/128250576.py:14: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Discussion", height=500)
/tmp/ipykernel_4300/128250576.py:14: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Discussion", height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c1aff761b1456a8c98.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c1aff761b1456a8c98.gradio.live


In [17]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [44]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File
from typing import List
from pydantic import BaseModel
from typing import List, Tuple
from pyngrok import ngrok
from fastapi import Response, Form
import uvicorn

# --- INITIALISATION API ---
app = FastAPI(title="Smart HR Assistant API")

# Structure des données attendues par l'API
class ChatRequest(BaseModel):
    message: str
    chat_history: List[Tuple[str, str]] = []

# --- LOGIQUE DE L'API ---

@app.post("/ask")
async def ask_hr(request: ChatRequest):
    # fonction 'respond' renvoie ("", chat_history)
    # récupère seulement le dernier message de l'IA ajouté à l'historique
    _, updated_history = respond(request.message, request.chat_history)

    # la dernière réponse de l'IA
    last_ai_response = updated_history[-1][1]

    return {
        "answer": last_ai_response,
        "history": updated_history
    }

class GradioSim:
    def __init__(self, path):
        self.name = path

@app.post("/upload")
async def api_upload(uploaded_file: UploadFile = File(...)):
    # 1. créer le fichier physique dans /tmp pour avoir un .name
    temp_path = f"/tmp/{uploaded_file.filename}"
    with open(temp_path, "wb") as buffer:
        shutil.copyfileobj(uploaded_file.file, buffer)

    # 2. transformer en format compatible
    fake_gradio_list = [GradioSim(temp_path)]
    resultat = upload_files(fake_gradio_list)

    return {"message": resultat}

@app.post("/clear_db")
async def api_clear():
    resultat = vider_db()
    return {"message": resultat}

@app.post("/whatsapp")
async def reply_whatsapp(Body: str = Form(...), From: str = Form(...)):
    # 1. récupérer la question de l'utilisateur (Body)
    question = Body

    # 2. appel respond

    _, updated_history = respond(question, [])
    reponse_ia = updated_history[-1][1]

    # 3. renvoyer la réponse à whatsapp
    twiml_response = f"""<?xml version="1.0" encoding="UTF-8"?>
    <Response>
        <Message>{reponse_ia}</Message>
    </Response>"""
    print("réponse: ", reponse_ia)
    return Response(content=twiml_response, media_type="application/xml")


@app.get("/")
def health_check():
    return {"status": "L'assistant RH est en ligne"}

# --- LANCEMENT DU TUNNEL NGROK ---
ngrok.set_auth_token("3BfuT9wRmvQsudT36LrikrH4ktH_7mC7yjqn2jn7WvNqkDVQu")

nest_asyncio.apply()
try:
    public_url = ngrok.connect(8000)
    print(f"API EN LIGNE : {public_url}")
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()
except Exception as e:
    print(f"Erreur : {e}")

API EN LIGNE : NgrokTunnel: "https://prenegligent-ophthalmological-malinda.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [4300]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2001:861:8c81:8b90:4c79:e94f:5cc:4c6e:0 - "GET / HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4300]
